## Retail AI Assistant (Smart-Shopper AI)
1. Overview

This project implements a simulation-based Retail AI Assistant capable of performing two roles:

* **Personal Shopper (Revenue Agent)** — recommends products based on user preferences and constraints
* **Customer Support Assistant (Operations Agent)** — evaluates return eligibility using defined business policies

The system follows a **hybrid agentic architecture**, where:

* A language model is used for **intent detection and explanation**
* Deterministic Python functions (tools) handle **data retrieval and business logic**

This design ensures both flexibility (natural language understanding) and reliability (rule-based decisions).



In [ ]:
## First of all , we need the important , necessary libraries
pip install pandas 

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install openai

  Using cached openai-2.32.0-py3-none-any.whl.metadata (31 kB)
Using cached openai-2.32.0-py3-none-any.whl (1.2 MB)
   ---------------------------------------- 0.0/204.7 kB ? eta -:--:--
   -- ------------------------------------- 10.2/204.7 kB ? eta -:--:--
   ----- --------------------------------- 30.7/204.7 kB 325.1 kB/s eta 0:00:01
   ----------- --------------------------- 61.4/204.7 kB 465.5 kB/s eta 0:00:01
   ----------------- --------------------- 92.2/204.7 kB 581.0 kB/s eta 0:00:01
   -------------------------------- ----- 174.1/204.7 kB 748.1 kB/s eta 0:00:01
   -------------------------------------- 204.7/204.7 kB 777.7 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from datetime import datetime
from openai import OpenAI
import json

## 2. System Architecture

The architecture is divided into three main layers:



### 2.1 Data Layer

The system uses three data sources:

* **Product Inventory CSV** — product attributes, tags, stock, pricing
* **Orders CSV** — purchase records
* **Policy File** — return and exchange rules

These are loaded into memory using Pandas and accessed exclusively through tool functions.

In [2]:
products = pd.read_csv("C:/Users/Dell/Desktop/Retail_AI_Agent/datasets/product_inventory.csv")
orders = pd.read_csv("C:/Users/Dell/Desktop/Retail_AI_Agent/datasets/orders.csv")

with open("C:/Users/Dell/Desktop/Retail_AI_Agent/datasets/policy.txt", "r") as f:
    policy_text = f.read()

print("Products:", len(products))
print("Orders:", len(orders))

Products: 100
Orders: 100


## 2.2 Tool Layer (Deterministic Functions)

The system defines four core tools:

* `search_products(filters)`
* `get_product(product_id)`
* `get_order(order_id)`
* `evaluate_return(order_id)`

Each tool:

* Operates on real CSV datasets
* Applies deterministic filtering or rule-based logic
* Returns structured data

#### Product Recommendation Logic

The `search_products` tool:

* Filters by size availability
* Applies price constraints
* Prioritizes sale items
* Uses `bestseller_score` for ranking

#### Return Evaluation Logic

The `evaluate_return` tool enforces business rules such as:

* Clearance items → not returnable
* Sale items → limited return window (store credit only)
* Vendor exceptions (e.g., exchange-only or extended window)
* Standard return window → 14 days

All decisions are computed programmatically, ensuring consistency.

In [3]:
## Creating the function to search the products 
def search_products(filters: dict):
    df = products.copy()

    if "size" in filters:
        df = df[df["sizes_available"].astype(str).str.contains(str(filters["size"]))]

    if "max_price" in filters:
        df = df[df["price"] <= filters["max_price"]]

    if filters.get("on_sale"):
        df = df[df["is_sale"] == True]

    if "tags" in filters:
        for tag in filters["tags"]:
            df = df[df["tags"].str.contains(tag, case=False, na=False)]

    df = df.sort_values(by="bestseller_score", ascending=False)

    return df.head(5).to_dict(orient="records")

In [4]:
# Creatig the function to fetch the product 
def get_product(product_id: int):
    result = products[products["product_id"] == product_id]
    if result.empty:
        return None
    return result.iloc[0].to_dict()

In [5]:
## Creating the function to get the specific order on the basis of order_id
def get_order(order_id: int):
    result = orders[orders["order_id"] == order_id]
    if result.empty:
        return None
    return result.iloc[0].to_dict()

In [6]:
## The Function , we used here to evaluate the returns 
def evaluate_return(order_id: int):
    order = get_order(order_id)
    if not order:
        return {"error": "Order not found"}

    product = get_product(order["product_id"])
    if not product:
        return {"error": "Product not found"}

    order_date = datetime.strptime(order["order_date"], "%Y-%m-%d")
    days_since = (datetime.now() - order_date).days

    # Clearance
    if product.get("is_clearance"):
        return {
            "eligible": False,
            "reason": "Clearance items are final sale"
        }

    # Sale
    if product.get("is_sale"):
        if days_since <= 7:
            return {
                "eligible": True,
                "type": "store_credit"
            }
        return {
            "eligible": False,
            "reason": "Sale return window expired"
        }

    # Vendor rules
    if product.get("vendor") == "Aurelia Couture":
        return {
            "eligible": True,
            "type": "exchange_only"
        }

    if product.get("vendor") == "Nocturne":
        if days_since <= 21:
            return {
                "eligible": True,
                "type": "refund"
            }
        return {"eligible": False}

    # Normal items
    if days_since <= 14:
        return {
            "eligible": True,
            "type": "refund"
        }

    return {
        "eligible": False,
        "reason": "Return window expired"
    }

In [7]:
def call_tool(name, args):
    print(f"\n TOOL CALLED: {name} with {args}\n")

    if name == "search_products":
        return search_products(args)
    elif name == "get_product":
        return get_product(**args)
    elif name == "get_order":
        return get_order(**args)
    elif name == "evaluate_return":
        return evaluate_return(**args)

In [8]:
client = OpenAI()

SYSTEM_PROMPT = """
You are a retail AI assistant.

You have access to tools:
- search_products
- get_product
- get_order
- evaluate_return

Rules:
- NEVER guess data
- ALWAYS use tools when needed
- If data not found → say clearly
- Explain reasoning step-by-step

Shopping:
- Consider size, stock, price, sale, bestseller_score

Support:
- Apply return policy strictly
"""

In [9]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": "Search products",
            "parameters": {
                "type": "object",
                "properties": {
                    "size": {"type": "string"},
                    "max_price": {"type": "number"},
                    "on_sale": {"type": "boolean"},
                    "tags": {"type": "array", "items": {"type": "string"}}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_order",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "integer"}
                },
                "required": ["order_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "evaluate_return",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "integer"}
                },
                "required": ["order_id"]
            }
        }
    }
]



### 2.3 Agent Layer (LLM + Orchestration)

The agent is implemented using a Groq-hosted LLM (`llama-3.1-8b-instant`) and performs:

* Intent classification (shopping vs return)
* Response generation (human-like explanations)
* Orchestration of tool calls

Since Groq does not natively support structured tool calling, a **manual orchestration layer** is implemented:

* The LLM determines intent
* Python logic selects and executes the appropriate tool
* The LLM then explains the tool output

This creates a controlled and interpretable agent pipeline.


In [11]:
pip install Groq

  Using cached groq-1.2.0-py3-none-any.whl.metadata (16 kB)
Using cached groq-1.2.0-py3-none-any.whl (142 kB)
Note: you may need to restart the kernel to use updated packages.


In [12]:
from groq import Groq

client = Groq(api_key="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")

In [13]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Hello"}]
)

print(response.choices[0].message.content)

Hello. Is there something I can help you with, or would you like to chat?


In [14]:
from groq import Groq
import re

client = Groq(api_key="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")

def run_agent(user_input):
    print(f"\nUser: {user_input}\n")

    # Step 1 — Intent detection using LLM
    intent_prompt = f"""
Classify the user request into one of:
1. shopping
2. return

User input: {user_input}

Respond with ONLY one word: shopping OR return
"""

    intent_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": intent_prompt}]
    )

    intent = intent_response.choices[0].message.content.strip().lower()

    print(f" Detected Intent: {intent}\n")

    # Step 2 — Handle RETURN flow
    if "return" in intent:
        print(" TOOL CALLED: evaluate_return\n")

        match = re.search(r'\d+', user_input)
        if not match:
            return "Please provide a valid order ID."

        order_id = int(match.group())

        result = evaluate_return(order_id)

        if "error" in result:
            return "I could not find that order. Please check the order ID."

        # Use LLM to explain result
        explanation_prompt = f"""
User asked: {user_input}

Return evaluation result:
{result}

Explain clearly to the user with reasoning.
"""

        explanation = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": explanation_prompt}]
        )

        return explanation.choices[0].message.content

    # Step 3 — Handle SHOPPING flow
    else:
        print(" TOOL CALLED: search_products\n")

        # Basic filter extraction (you can improve this)
        filters = {
            "size": "8" if "8" in user_input else None,
            "max_price": 300 if "300" in user_input else None,
            "on_sale": "sale" in user_input.lower(),
            "tags": ["modest"] if "modest" in user_input.lower() else []
        }

        # Clean None values
        filters = {k: v for k, v in filters.items() if v}

        results = search_products(filters)

        if not results:
            return "I couldn’t find any products matching your criteria."

        # Use LLM to explain recommendations
        explanation_prompt = f"""
User request: {user_input}

Products found:
{results}

Explain why these products are good recommendations based on:
- size
- price
- sale status
- bestseller score
"""

        explanation = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": explanation_prompt}]
        )

        return explanation.choices[0].message.content

In [13]:
def run_agent(user_input):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
        tool_choice="auto"
    )

    msg = response.choices[0].message

    # If tool is called
    if msg.tool_calls:
        tool_call = msg.tool_calls[0]

        tool_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        print(f"\n TOOL CALLED: {tool_name} with {args}\n")

        result = call_tool(tool_name, args)

        # Append assistant tool call
        messages.append({
            "role": "assistant",
            "tool_calls": msg.tool_calls
        })

        # Append tool response (FIXED: tool_call_id added)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

        # Final response
        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )

        return final_response.choices[0].message.content

    return msg.content

In [16]:
print(run_agent("I need a modest evening gown under 300 in size 7 on sale"))


User: I need a modest evening gown under 300 in size 7 on sale

 Detected Intent: shopping

 TOOL CALLED: search_products



Based on the user request for a modest evening gown under 300 in size 7, I will evaluate each product from the provided list of products.

1. **Lumiere Style 28 (P0028)**
- Size: Unfortunately, this gown is not available in size 7. It's available in sizes 4, 6, 8, 12, 14, and 16. (Not a match)
- Price: 226, which is under 300. (Match)
- Sale status: Yes, it's on sale. (Match)
- Bestseller score: 97, which suggests it's a very popular and well-rated product. (Good indicator of product quality)

2. **Lumiere Style 4 (P0004)**
- Size: Unfortunately, this gown is not available in size 7. It's available in sizes 6, 8, 14. (Not a match)
- Price: 120, which is under 300. (Match)
- Sale status: Yes, it's on sale. (Match)
- Bestseller score: 90, which suggests it's a popular and well-rated product. (Good indicator of product quality)

3. **Nocturne Style 56 (P0056)**
- Size: Unfortunately, this gown is not available in size 7. It's available in sizes 4, 6, 8, 10, 12, 14, 16. (Not a match)
- Pri

In [17]:
print(run_agent("Show me casual dresses size 6 under 200"))


User: Show me casual dresses size 6 under 200

 Detected Intent: shopping

 TOOL CALLED: search_products

Based on the given criteria, here are some good product recommendations that are suitable for a casual dress size 6 under $200:

1. **Lumiere Style 62 (P0062)** - This product is a good recommendation because it meets all the criteria.
- **Size**: The dress is available in size 6.
- **Price**: The current price is $181, which is under the budget of $200.
- **Sale status**: The product is on sale, indicating that it may offer a discount.
- **Bestseller score**: The product has a bestseller score of 96, which indicates that it's a popular product among customers.

2. **Aurelia Couture Style 3 (P0003) is not a suitable match due to the high cost ($263), but Aurelia Couture Style 38 (P0038)** - The product is a good recommendation because it meets some of the criteria.
- **Price**: The current price is $163, which is under the budget of $200.
- **Sale status**: The product is on sale.

In [18]:
print(run_agent("Order O006 — can I return this?"))


User: Order O006 — can I return this?

 Detected Intent: return

 TOOL CALLED: evaluate_return

I could not find that order. Please check the order ID.


In [19]:
print(run_agent("Order O0006 — can I return this?"))


User: Order O0006 — can I return this?

 Detected Intent: return

 TOOL CALLED: evaluate_return

I could not find that order. Please check the order ID.
